# 04 — Coral Bleaching Analysis (Phase 3)

Binary NOAA CNN + Whiteness-Guided Bleaching Severity (WGBS) → 0-100 index and 5-class condition (Healthy/Mild/Moderate/Severe/Dead).

In [ ]:
# --- Environment check -------------------------------------------------------
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
PROJECT = pathlib.Path("/content/reefscan") if IN_COLAB else pathlib.Path.cwd()
print("Colab:", IN_COLAB, "| project:", PROJECT)
if IN_COLAB:
    import os
    from google.colab import drive
    # Mount Drive so datasets/checkpoints survive session resets (optional).
    # drive.mount("/content/drive")


In [ ]:
!python train.py --task coral --model all

In [ ]:
# WGBS on a few test patches
import glob, cv2, torch
from src.realtime.pipeline import load_best_classifier
from src.coral_analysis.severity import BleachingAssessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, classes, _, size = load_best_classifier("coral", "models", device)
assessor = BleachingAssessor()

for path in sorted(glob.glob("datasets/coral/imagefolders/test/*/*.jpg"))[:6]:
    img = cv2.imread(path)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    from PIL import Image
    from torchvision import transforms as T
    t = T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])(
        Image.fromarray(rgb).resize((size, size)))[None].to(device)
    import torch.nn.functional as F
    with torch.no_grad():
        probs = F.softmax(model(t), 1)[0].cpu().numpy()
    p_bl = float(probs[classes.index("CORAL_BL")])
    verdict = assessor.assess(img, p_bl)
    print(f"{path.split('/')[-2]:>8} | P(bleached)={p_bl:.2f} | BSI={verdict.index:5.1f} | {verdict.label}")
